In [1]:
import pandas as pd
from tableone import TableOne
from pymatch.Matcher import Matcher
import numpy as np
import matplotlib.pyplot as plt

import warnings

warnings.filterwarnings("ignore")

%matplotlib inline

In [2]:
df = pd.read_csv("../data/merged.csv")
df.set_index("case_number", inplace=True)

In [3]:
lookup = pd.read_csv("../data/drug_column_lookup.csv")
drugs = pd.read_csv("../data/drugs.csv")
column_map = {
    item["row_name"]: item["drug_name"] for item in lookup.to_dict(orient="records")
}
drugs.rename(columns=column_map, inplace=True)
targets = drugs[
    sorted(
        [c for c in drugs.columns if "fentanyl" in c.lower() or "heroin" in c.lower()]
        + ["case_number"]
    )
]
filtered_targets = targets.loc[:, targets.sum() > 0]
filtered_targets.set_index("case_number", inplace=True)
filtered_targets.rename(
    columns={c: f"{c}-ccmeo" for c in filtered_targets.columns}, inplace=True
)

In [4]:
df = pd.merge(
    left=df,
    right=filtered_targets,
    left_index=True,
    right_index=True,
    how="left",
    validate="1:1",
)

In [5]:
heroins = ["Heroin probable-ccmeo", "heroin-ccmeo", "HEROIN"]
fentanyl = ["fentanyl-ccmeo", "FENTANYL"]
analogs = [
    "3-methylfentanyl-ccmeo",
    "Acrylfentanyl-ccmeo",
    "Fluoroisobutyryl fentanyl-ccmeo",
    "Heroin probable-ccmeo",
    "acetyl fentanyl-ccmeo",
    "benzyl fentanyl-ccmeo",
    "butyryl fentanyl-ccmeo",
    "cyclopropyl fentanyl-ccmeo",
    "fentanyl-ccmeo",
    "fluoro fentanyl-ccmeo",
    "fluoro furanyl fentanyl-ccmeo",
    "heroin-ccmeo",
    "isovaleryl fentanyl-ccmeo",
    "o_fentanyl_analogue-ccmeo",
    "para-fluoro furanyl fentanyl-ccmeo",
    "pivaloyl fentanyl-ccmeo",
    "valeryl fentanyl-ccmeo",
    "FENTANYL_meta",
]
target_cols = heroins + fentanyl + analogs
# ge = greater than or equal to
# gt = greater than
# greater than or equal to 1 because if any are true
df["is_heroin"] = df[heroins].gt(0).sum(axis=1).ge(1)
df["is_fentanyl"] = df[fentanyl].gt(0).sum(axis=1).ge(1)
df["is_analog"] = df[analogs].gt(0).sum(axis=1).ge(1)
df = df.loc[df["age"].str.contains("years")]
df.sample()

,ccmeo_year,body_rcvd_date,body_rcvd_date_day,body_rcvd_date_year,coded_mode,rc_vautopsy_type,cause_of_death,street,city,state,...,fluoro furanyl fentanyl-ccmeo,heroin-ccmeo,isovaleryl fentanyl-ccmeo,o_fentanyl_analogue-ccmeo,para-fluoro furanyl fentanyl-ccmeo,pivaloyl fentanyl-ccmeo,valeryl fentanyl-ccmeo,is_heroin,is_fentanyl,is_analog
case_number,,,,,,,,,,,,,,,,,,,,,
870,7,7.0,6.0,2020.0,HAMOVE---,POST,"Acute fentanyl, morphine, and citalopram toxic...",12 Park Place,Bedford,Ohio,...,0,0,0,0,0,0,0,False,True,True


In [6]:
df.loc[:, "Home":"Fifth floor"] = df.loc[:, "Home":"Fifth floor"].fillna(0)
print(f'Total nulls: {df.loc[:, "Home": "Fifth floor"].isna().sum().sum()}')
print("Counts:")
df.loc[:, "Home":"Fifth floor"].sum().to_dict()

Total nulls: 0
Counts:


{'Home': 514.0,
 'Apartment block': 31.0,
 'Bathroom': 141.0,
 'House': 90.0,
 'Apartment': 238.0,
 'Bedroom': 451.0,
 'Dining room': 33.0,
 'Kitchen': 166.0,
 'Upper floor': 2.0,
 'Boarding house': 4.0,
 'Third floor': 11.0,
 'Upstairs': 46.0,
 'Hallway': 32.0,
 'Single room': 5.0,
 'Basement floor': 26.0,
 'Residential home': 4.0,
 'Second floor': 18.0,
 'Back yard': 8.0,
 'Front yard': 4.0,
 'Apartment in apartment building': 2.0,
 'Ground floor': 6.0,
 'Garage': 23.0,
 'Downstairs': 11.0,
 'Landing': 3.0,
 'Bottom of staircase': 5.0,
 'First floor': 17.0,
 'Side room': 3.0,
 'Seventh floor': 1.0,
 'Detached house': 1.0,
 'Shower room': 1.0,
 'Sitting room': 10.0,
 'Top of staircase': 1.0,
 'Eighth floor': 1.0,
 'Castle': 1.0,
 'Fifth floor': 1.0}

In [15]:
outside_cols = [
    "Garage",
    "Detached house",
    "Back yard",
    "Front yard",
]
room_cols = [
    "Single room",
    "Side room",
    "Shower room",
    "Sitting room",
    "Bathroom",
    "Bedroom",
    "Dining room",
    "Kitchen",
    "Hallway",  # currently included to reduce dimensionality
]
df["outside"] = df[outside_cols].gt(0).sum(axis=1).ge(1)
df["room"] = df[room_cols].gt(0).sum(axis=1).ge(1)
# currently included this in rooms
#  df["hallway"] = df["Hallway"] > 0
#  largest category, other categories are subsets of this
#  further, rows that don't have this aren't in "home premises"
df["home"] = df["Home"] > 0


# doesn't include hallway for now
def categorize_location(row) -> str:
    outside = row["outside"]
    room = row["room"]
    home = row["home"]
    if outside and room:
        return "outside & room"
    elif outside and not room:
        return "outside"
    elif not outside and room:
        return "room"
    elif home:
        return "home"
    else:
        return "not-home"


df["location"] = df.apply(categorize_location, axis=1)

In [16]:
def categorize_age(x: str) -> str:
    number = int(x.split(" ")[0])
    if number < 18:
        return "<17"
    elif number < 25:
        return "18-24"
    elif number < 35:
        return "25-34"
    elif number < 45:
        return "35-44"
    elif number < 55:
        return "45-54"
    elif number < 65:
        return "55-64"
    else:
        return "65+"


df["age_cat"] = df["age"].apply(categorize_age)
df["age_cat"].value_counts()

55-64    276
35-44    245
25-34    235
45-54    221
65+       99
18-24     49
<17        7
Name: age_cat, dtype: int64

In [17]:
def convert_ccmeo_year(x: int) -> int:
    if x == 6:
        return 2019
    elif x == 7:
        return 2020
    else:
        raise ValueError(f"Unknown CCME Year: {x}")


df["year"] = df["ccmeo_year"].apply(convert_ccmeo_year)
df["year"].value_counts()

2019    581
2020    551
Name: year, dtype: int64

In [18]:
useful_cols = [
    "is_heroin",
    "is_fentanyl",
    "is_analog",
    "year",
    "age_cat",
    "race",
    "gender",
]
# location_cols = ["outside", "room", "home"]
location_cols = ["location"]
dff = df[useful_cols + location_cols].reset_index()
dff.replace({0: False, 1: True}, inplace=True)
dff.sample()

,case_number,is_heroin,is_fentanyl,is_analog,year,age_cat,race,gender,location
873,876,False,True,True,2020,55-64,White,Male,not-home


At this point we are all prepared and can match on demographics...

In [19]:
# MATCHING --> pymatch

In [20]:
# TABLE ONE --> tableone

In [21]:
table1 = TableOne(
    data=dff,
    columns=useful_cols,
    categorical=useful_cols,
    groupby="location",
    pval=True,
)
print(table1.tabulate(tablefmt="github"))

|                    |                                   | Missing   | Overall    | home       | not-home   | outside   | outside & room   | room       | P-Value   |
|--------------------|-----------------------------------|-----------|------------|------------|------------|-----------|------------------|------------|-----------|
| n                  |                                   |           | 1132       | 268        | 445        | 20        | 4                | 395        |           |
| is_heroin, n (%)   | False                             | 0         | 604 (53.4) | 180 (67.2) | 231 (51.9) | 10 (50.0) | 2 (50.0)         | 181 (45.8) | <0.001    |
|                    | True                              |           | 528 (46.6) | 88 (32.8)  | 214 (48.1) | 10 (50.0) | 2 (50.0)         | 214 (54.2) |           |
| is_fentanyl, n (%) | False                             | 0         | 340 (30.0) | 107 (39.9) | 125 (28.1) | 4 (20.0)  | 2 (50.0)         | 102 (25.8) | 0.001     |
|   

In [249]:
table1.to_excel("table1.xlsx")